In [37]:
import json
import pandas as pd
import numpy as np

In [38]:
weights = json.load(open('./data/origin_dest_weights.json', 'r'))
pairs = json.load(open('./data/origin_pairs.json', 'r'))
tract_dem = json.load(open('data/tract_demographics.json', 'r'))

In [39]:
results = json.load(open('./computation_results/weighted/final-1-2.json', 'r')) | json.load(open('./computation_results/weighted/final-2-2.json', 'r'))

In [43]:
df = pd.DataFrame(index=list(results.keys()), columns=['majority', 'shortest_path', 'shortest_restricted_path', 'relative_increase', 'absolute_increase', 'n_cameras'])

for origin_tract, dest_tracts in results.items():
    
    if dest_tracts == {}:
        df.drop(origin_tract, axis=0, inplace=True)
        continue

    total_weights = []
    if not origin_tract in tract_dem:
        df.drop(origin_tract, axis=0, inplace=True)
        continue
    
    majority = tract_dem[origin_tract]
    
    for d, v in dest_tracts.items():
        if not v: continue
        total_weights.append(weights[origin_tract][d])
        
    total_weights = np.array(total_weights)/np.sum(total_weights)
    print(total_weights)
    budgets = []
    n_cameras = []
    unrestricted_time_to_reach = []
    restricted_time_to_reach = []

    for d, v in dest_tracts.items():
        if not v: continue
        budgets = list(v.keys())
        highest, lowest = (budgets[0], budgets[-1])

        n_cameras.append(int(highest))

        unrestricted_time_to_reach.append(v[highest])
        restricted_time_to_reach.append(v[lowest])

    
    w_uttr = unrestricted_time_to_reach @ total_weights
    w_rttr = restricted_time_to_reach @ total_weights
    
    df.loc[df.index == origin_tract] = [
        majority,
        w_uttr,
        w_rttr,
        (w_rttr - w_uttr)/w_uttr,
        w_rttr - w_uttr,
        total_weights @ n_cameras
    ]

df.relative_increase = df.relative_increase.map(lambda x: 0 if pd.isna(x) else x)
df.to_csv('./analysis/weighted.csv')

[0.5 0.5]
[1.]
[0.05870031 0.02938099 0.25885802 0.51204826 0.14101243]
[0.05931225 0.0805169  0.06378223 0.06190513 0.05992039 0.35191931
 0.06015069 0.06015069 0.0805169  0.06190513 0.05992039]
[0.18659933 0.01097937 0.01056182 0.01105339 0.12299664 0.0269768
 0.0105739  0.02138311 0.0483888  0.02125187 0.0105739  0.03367065
 0.20298216 0.14305354 0.09456799 0.01071609 0.03367065]
[0.23871737 0.76128263]
[1.]
[0.03190884 0.01593792 0.20433259 0.14232953 0.01593792 0.01593792
 0.1021663  0.01593792 0.12305329 0.01593792 0.31651984]
[0.07081443 0.06961038 0.23173292 0.06958042 0.06961038 0.06958042
 0.20997527 0.06958042 0.13951535]
[0.43000704 0.1416608  0.1416608  0.28667136]
[0.26034064 0.26034064 0.26034064 0.10948903 0.10948903]
[1.]
[0.16791468 0.16791468 0.16791468 0.49625595]
[0.02959404 0.02959404 0.02959404 0.02959404 0.02959404 0.52853553
 0.02963245 0.26426776 0.02959404]
[0.08680315 0.26130711 0.0860455  0.3074552  0.08625956 0.17212948]
[0.12997268 0.13041173 0.13074743 0

In [44]:
df = pd.DataFrame(index=list(results.keys()), columns=['majority', 'shortest_path', 'shortest_restricted_path', 'relative_increase', 'absolute_increase', 'n_cameras'])

for origin_tract, dest_tracts in results.items():
    
    if dest_tracts == {}:
        df.drop(origin_tract, axis=0, inplace=True)
        continue

    total_weights = []
    if not origin_tract in tract_dem:
        df.drop(origin_tract, axis=0, inplace=True)
        continue
    
    majority = tract_dem[origin_tract]
    
    for d, v in dest_tracts.items():
        if not v: continue
        total_weights.append(weights[origin_tract][d])
        
    total_weights = np.array(total_weights)/np.sum(total_weights)
    print(total_weights)
    budgets = []
    n_cameras = []
    unrestricted_time_to_reach = []
    restricted_time_to_reach = []

    for d, v in dest_tracts.items():
        if not v: continue
        budgets = list(v.keys())
        highest, lowest = (budgets[0], budgets[-1])

        n_cameras.append(int(highest))

        unrestricted_time_to_reach.append(v[highest])
        restricted_time_to_reach.append(v[lowest])

    
    w_uttr = unrestricted_time_to_reach @ total_weights
    w_rttr = restricted_time_to_reach @ total_weights
    if total_weights @ n_cameras == 0: df.drop(origin_tract, axis=0, inplace=True)
    df.loc[df.index == origin_tract] = [
        majority,
        w_uttr,
        w_rttr,
        (w_rttr - w_uttr)/w_uttr,
        w_rttr - w_uttr,
        total_weights @ n_cameras
    ]

df.relative_increase = df.relative_increase.map(lambda x: 0 if pd.isna(x) else x)
df.to_csv('./analysis/nonzero-weighted.csv')

[0.5 0.5]
[1.]
[0.05870031 0.02938099 0.25885802 0.51204826 0.14101243]
[0.05931225 0.0805169  0.06378223 0.06190513 0.05992039 0.35191931
 0.06015069 0.06015069 0.0805169  0.06190513 0.05992039]
[0.18659933 0.01097937 0.01056182 0.01105339 0.12299664 0.0269768
 0.0105739  0.02138311 0.0483888  0.02125187 0.0105739  0.03367065
 0.20298216 0.14305354 0.09456799 0.01071609 0.03367065]
[0.23871737 0.76128263]
[1.]
[0.03190884 0.01593792 0.20433259 0.14232953 0.01593792 0.01593792
 0.1021663  0.01593792 0.12305329 0.01593792 0.31651984]
[0.07081443 0.06961038 0.23173292 0.06958042 0.06961038 0.06958042
 0.20997527 0.06958042 0.13951535]
[0.43000704 0.1416608  0.1416608  0.28667136]
[0.26034064 0.26034064 0.26034064 0.10948903 0.10948903]
[1.]
[0.16791468 0.16791468 0.16791468 0.49625595]
[0.02959404 0.02959404 0.02959404 0.02959404 0.02959404 0.52853553
 0.02963245 0.26426776 0.02959404]
[0.08680315 0.26130711 0.0860455  0.3074552  0.08625956 0.17212948]
[0.12997268 0.13041173 0.13074743 0